# MTL-LymphNet v2 -- Kaggle / Colab Runner

This is a **thin execution notebook** for the modular `mtl_lymphnet` package. All science/engineering logic lives in `src/mtl_lymphnet/`; this notebook only:

1. Installs the package and its dependencies.
2. Resolves the environment (Kaggle / Colab / local) via `mtl_lymphnet.env`.
3. Calls the framework's CLI (`mtl-lymphnet ...`) to run experiments, statistics, calibration, and export tables/figures.

### Reviewer-driven fixes implemented
- **R1.1** -- leakage-safe nested train/val/test cross-validation
- **R1.4** -- backbone-agnostic framework (no single backbone claimed "best"); this notebook trains and externally validates all three registered backbones and presents the trade-off directly (Section 3b, Section 4)
- **R1.5** -- per-epoch loss-weight trajectory export; all four weighting strategies (Uncertainty, Fixed 0.5/0.5, Fixed 0.7/0.3, DWA) are externally validated, not just the one used in the main run (Section 3c)
- **R1.6** -- multi-seed internal CV (3 seeds) AND multi-seed external validation (3 seeds, every configuration in Section 3); bootstrap/DeLong confidence intervals computed automatically for every external configuration (`stats_external_ci.csv`, Section 5)
- **R1.7** -- calibration analysis (ECE/MCE/reliability diagrams) and explicit threshold policy, covering every external configuration automatically (Section 5)
- **R1.11** -- standardized inference benchmarking (hardware, precision, warm-up, repetitions), automatically run for all three backbones on CPU and GPU (Section 6)

### Before running
1. Upload/clone the project repo (so `pyproject.toml` and `dataset/` are available)
2. If a Hugging Face token is needed, set it via Kaggle Secrets as `HF_TOKEN` -- never hardcode it
3. Delete `outputs_v2/exp1_results.pkl` and `outputs_v2/experiment_results.pkl` if you want a completely fresh run (note: this also clears any accumulated backbone/strategy sweep results from Section 3b/3c, since they live in the same `experiment_results.pkl` under `exp2_variants`)
4. This notebook runs a full experiment matrix -- Section 3 alone is ~36 external-validation training runs, in addition to the 15-run internal CV in Section 2. Budget compute time accordingly. Section 0 below runs a fast CPU-only smoke test first to catch setup/import errors before committing to the full run.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import sys
from kaggle_secrets import UserSecretsClient

# 1. Get token from Kaggle Secrets
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

# 2. Define repository details
# Replace 'your-username' and 'repo-name' with your own
GITHUB_USERNAME = "muhammad-thahiruddin"
REPO_NAME = "mtl_lymphnet"

# 3. Construct URL dynamically injecting the auth token
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 4. Clone repository to Kaggle local directory
!git clone {repo_url}

# 5. (Optional) Navigate into the newly cloned repo folder
import sys
sys.path.append(f"./{REPO_NAME}")

In [ ]:
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ

# Adjust this to wherever the project folder (containing pyproject.toml) lives
base_dir = os.environ.get('MTL_LYMPHNET_PROJECT_DIR', os.getcwd())
PROJECT_DIR = os.path.join(base_dir, REPO_NAME)

print(f'Kaggle={IS_KAGGLE}  Colab={IS_COLAB}  PROJECT_DIR={PROJECT_DIR}')
%cd {PROJECT_DIR}
%pip install -e . -q

In [ ]:
PROJECT_DIR = '/kaggle/working/mtl_lymphnet'
%cd {PROJECT_DIR}

SRC_DIR = os.path.join(PROJECT_DIR, 'src')

# 1. Use insert(0, ...) INSTEAD of append, prioritizing this repo
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR) 

# 2. Clear previous module cache (forces Python to reload modules)
keys_to_delete = [k for k in sys.modules.keys() if k.startswith('mtl_lymphnet')]
for k in keys_to_delete:
    del sys.modules[k]

print("Primary search path:", sys.path[0])

# 3. Re-attempt imports
from mtl_lymphnet.env import setup_environment, print_environment_summary

cfg, env = setup_environment()
print_environment_summary(cfg, env)

## 0. Smoke test (recommended before the full run)

Quick, CPU-friendly sanity check (2 folds, 2 epochs, a tiny subset of the data) that exercises the full leakage-safe pipeline end-to-end -- nested split, training, held-out evaluation. Takes about 1-2 minutes even without a GPU, and catches import/runtime errors from the Step 1 code changes before you commit to the expensive full matrix below (which this codebase has only been verified statically, not executed -- see `docs/status_report_and_action_plan.md`, item 1.11).

In [ ]:
!mtl-lymphnet smoke-test

## 1. Dataset inventory (EDA)

Scans `dataset/Center 1` and `dataset/Center 2`, validates image/mask/lesion completeness, and saves an inventory CSV.

In [ ]:
!mtl-lymphnet eda

## 1.5 Clean previous results (optional -- only for a completely fresh run)

The framework is backbone-agnostic (R1.4): different backbones and loss-weighting strategies are tracked separately under `exp2_variants` / per-backbone ablation keys (Bug B1 fix) and do NOT overwrite each other, so you do NOT need to clean up between backbones or strategies. Only run the cell below if you want to discard ALL previous results (internal CV, every external-validation configuration, every ablation) and start completely over.

In [ ]:
import os
for f in ['outputs_v2/exp1_results.pkl', 'outputs_v2/experiment_results.pkl']:
    if os.path.exists(f):
        os.remove(f)
        print(f'Deleted {f}')
    else:
        print(f'{f} not found (clean start)')

## 2. Experiment 1 -- Internal cross-validation (multi-seed, leakage-safe)

**Backbone:** TinyViT-5M (default/reference configuration -- not claimed to be superior; see Section 3b and Section 4 for the full three-backbone trade-off comparison, R1.4)

**R1.1 fix:** Within each outer fold, the training partition is split 80/20 into an inner train/val set. Early stopping and checkpoint selection use only the inner validation split; the held-out outer fold is evaluated exactly once, after training.

**R1.6 fix:** Runs with 3 random seeds (42, 43, 44) to account for training variability.  
Total: 5 folds x 3 seeds = **15 runs**.

In [ ]:
!mtl-lymphnet --config configs/exp1_internal_cv.yaml run --experiment internal_cv --seeds 42,43,44 --force-rerun-exp1

## 3. Experiment 2 -- Bidirectional external (cross-center) validation

Also leakage-safe: the training center is split into an inner train/val (early stopping only); the external test center is evaluated exactly once and never influences model selection.

This section has three parts, all using the same leakage-safe protocol:
- **3a**: default configuration (TinyViT-5M, Uncertainty weighting), upgraded to 3 seeds -- the primary/headline result, stored under `exp2_c1_to_c2` / `exp2_c2_to_c1`.
- **3b**: backbone sweep -- MobileNetV3-Large and EfficientNet-B0, same protocol, so R1.4's backbone trade-off has external evidence, not just internal CV.
- **3c**: loss-weighting strategy sweep -- Fixed (0.5/0.5), Fixed (0.7/0.3), and DWA, so R1.5's request for "cross-center results for each weighting strategy" is answered directly.

Every configuration below is tracked separately, with its own checkpoint and TensorBoard run (Bug B1 fix), and is picked up automatically by `stats` / `calibration` / `export-tables` in Section 5 -- no extra flags needed there.

### 3a. Default configuration, upgraded to 3 seeds

No `--backbone` / `--loss-strategy` override, so this UPDATES the primary `exp2_c1_to_c2` / `exp2_c2_to_c1` result in place (not a separate variant).

In [ ]:
!mtl-lymphnet --config configs/exp2_external_validation.yaml run --experiment external_validation --seeds 42,43,44

### 3b. Backbone sweep (R1.4)

Same protocol as 3a, alternative backbones only (the default backbone's external result already comes from 3a above). Each backbone's result is kept separate under `exp2_variants` -- nothing here overwrites 3a or the other backbone.

In [ ]:
backbone_sweep = {
    "MobileNetV3-Large": "tu-mobilenetv3_large_100",
    "EfficientNet-B0": "tu-efficientnet_b0",
}
for name, timm_key in backbone_sweep.items():
    print(f"\n=== External validation backbone sweep: {name} ({timm_key}) ===")
    !mtl-lymphnet --config configs/exp2_external_validation.yaml run --experiment external_validation --backbone {timm_key} --seeds 42,43,44

### 3c. Loss-weighting strategy sweep (R1.5)

Same protocol, default backbone (TinyViT-5M), alternative loss-weighting strategies only (Uncertainty's external result already comes from 3a above). `--w-seg` is ignored by the `dwa` strategy internally but passed for CLI uniformity.

In [ ]:
strategy_sweep = [
    ("fixed", 0.5),
    ("fixed", 0.7),
    ("dwa", 0.5),
]
for strategy, w_seg in strategy_sweep:
    print(f"\n=== External validation strategy sweep: {strategy} (w_seg={w_seg}) ===")
    !mtl-lymphnet --config configs/exp2_external_validation.yaml run --experiment external_validation --loss-strategy {strategy} --w-seg {w_seg} --seeds 42,43,44

## 4. Ablation studies (Experiments 3-5) -- internal cross-validation only

These reuse Experiment 1's default-configuration results (TinyViT-5M, Uncertainty weighting) as one arm of each comparison, and train the remaining variants fresh via internal cross-validation (single seed per variant -- not upgraded to multi-seed; see `docs/status_report_and_action_plan.md` item 1.8 for why this is optional rather than required):

- **Experiment 3**: MTL (default config, reused from Exp 1) vs. Seg-Only vs. Cls-Only
- **Experiment 4**: backbone comparison -- TinyViT-5M (default, reused from Exp 1) vs. MobileNetV3-Large vs. EfficientNet-B0. This is the INTERNAL counterpart to the external backbone sweep in Section 3b -- together they give both internal and external evidence for R1.4's trade-off table.
- **Experiment 5**: loss-weighting strategy comparison -- Uncertainty (default, reused from Exp 1) vs. Fixed 0.5/0.5 vs. Fixed 0.7/0.3 vs. DWA. Internal counterpart to Section 3c.

In [ ]:
!mtl-lymphnet --config configs/exp3_ablation_mtl.yaml run --experiment ablation_mtl
!mtl-lymphnet --config configs/exp4_ablation_backbone.yaml run --experiment ablation_backbone
!mtl-lymphnet --config configs/exp5_ablation_loss.yaml run --experiment ablation_loss

## 5. Statistics, calibration, tables, and figures

- `stats`: Friedman + pairwise Wilcoxon (Dice) and DeLong (AUC) with Benjamini-Hochberg correction on internal CV, plus bootstrap CI on pairwise Dice differences; AND bootstrap/DeLong confidence intervals for AUC + Dice on EVERY external-validation configuration from Section 3a/3b/3c, saved to `stats_external_ci.csv` (R1.6)
- `calibration`: Expected/Maximum Calibration Error and reliability diagrams for internal CV and EVERY external-validation configuration from Section 3, not just the two primary directions (R1.7)
- `export-tables`: regenerates every manuscript table, including `table3_external_val.tex` with one row per Section 3 configuration -- tables never include the removed "Combined" score (R1.3)
- `export-figures`: regenerates figures from saved results. NOTE -- `fig_external_validation.png` still only plots the Section 3a default pair; it has not yet been redesigned to show every Section 3b/3c variant (deliberately deferred, see `docs/status_report_and_action_plan.md` item 1.9)

In [ ]:
RESULTS = f"{cfg.output_dir}/experiment_results.pkl"

!mtl-lymphnet stats --results {RESULTS}
!mtl-lymphnet calibration --results {RESULTS}
!mtl-lymphnet export-tables --results {RESULTS}
!mtl-lymphnet export-figures --results {RESULTS}

## 6. Inference benchmark (R1.11)

Standardized benchmark with hardware specs, batch size, precision, warm-up, and repetition count -- runs automatically for ALL THREE backbones, on CPU and GPU (when available), in a single call. Results saved to `benchmark_results.json` as a list (one entry per backbone x device), so re-running never overwrites a previous configuration's result.

In [ ]:
!mtl-lymphnet --config configs/exp1_internal_cv.yaml benchmark --batch-size 1 --precision fp32 --warmup 10 --reps 100

## 7. Package results for download

In [ ]:
import shutil
from datetime import datetime
from IPython.display import FileLink, display

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_base = f'experiment_{timestamp}'

shutil.make_archive(zip_base, 'zip', cfg.output_dir)
zip_path = f'{zip_base}.zip'
print(f'Archived {cfg.output_dir} -> {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)')

if IS_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    display(FileLink(zip_path))